# 07 — Interprétabilité : Grad-CAM (Phase 8, version allégée)

**À exécuter dans la même session Colab que `06_eval_finale.ipynb`, juste après** (recharge le checkpoint `outputs/models/resnet18_final.pt` sauvegardé par ce notebook — il ne survit pas à la fermeture de la session).

Question posée : le modèle regarde-t-il les **noyaux cellulaires** (morphologie) ou une **texture/teinte diffuse** liée à l'hôpital (voir `outputs/rapport/03_reference_simple.md` et `04_cnn.md`, qui posent la question sans la trancher) ? Version allégée, demandée par le retour du professeur : Grad-CAM sur quelques **faux négatifs et faux positifs de l'hôpital 1** (toujours l'hôpital des réglages, jamais celui du test — voir Phase 6).

Voir `PLAN.md` (Phase 8) et `outputs/rapport/08_interpretabilite.md`.

In [ ]:
# Colab/Kaggle : clone le depot si besoin et place le repertoire de travail dans notebooks/
# (sans effet en local : le depot est deja la, donc rien n'est clone ni deplace)
import os, subprocess, sys

REPO_URL = "https://github.com/YoucefLgr/Breast_Cancer.git"
EN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
EN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if EN_COLAB or EN_KAGGLE:
    repo_dir = "/content/Breast_Cancer" if EN_COLAB else "/kaggle/working/Breast_Cancer"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", REPO_URL, repo_dir], check=True)
    os.chdir(os.path.join(repo_dir, "notebooks"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)

print("Repertoire de travail :", os.getcwd())

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from src.utils import load_config, set_seed, get_device, detect_environment, save_metrics
from src.data import charger_sous_echantillon, PatchDataset, construire_transforms, ouvrir, HOPITAL_LABEL
from src.model import construire_modele
from src.train import predire_probas, choisir_seuil
from src.interpret import gradcam, selectionner_exemples

CFG = load_config()
SEED = CFG["seed"]
set_seed(SEED)
DEVICE = get_device()
VARIANTE_RETENUE = "base"   # Phase 5 : egalite statistique -> choix par defaut (voir Phase 6)
print("Environnement :", detect_environment(), "| device :", DEVICE)

## 1. Modèle final (rechargé, pas ré-entraîné)

In [ ]:
modele = construire_modele(CFG["modele"]["architecture"]).to(DEVICE)
modele.load_state_dict(torch.load("../outputs/models/resnet18_final.pt", map_location=DEVICE))
print("Checkpoint chargé.")

df = charger_sous_echantillon(CFG)
df_val = df[df.split == "val_ood"].reset_index(drop=True)
_, t_eval = construire_transforms(CFG, VARIANTE_RETENUE)
BS = CFG["modele"]["batch_size"]
NW = 2 if DEVICE == "cuda" else 0
dl_val = DataLoader(PatchDataset(df_val, t_eval), batch_size=BS, shuffle=False, num_workers=NW)

proba_val = predire_probas(modele, dl_val, DEVICE)
choix = choisir_seuil(df_val.label.values, proba_val, CFG["evaluation"]["rappel_cible"])
SEUIL = choix["seuil"]
print("Seuil (recalculé ici pour rester autonome) :", SEUIL)

## 2. Sélection des exemples (vrai/faux positif/négatif, 2 hôpitaux)

In [ ]:
df_val = df_val.assign(proba=proba_val)
pred_classe = (df_val.proba >= SEUIL).astype(int).values
# Priorite demandee par le prof : les erreurs (FN/FP) comptent plus que les succes
exemples = selectionner_exemples(
    df_val, pred_classe, n_par_categorie=1, seed=SEED,
    n_par_categorie_overrides={"faux_negatif": 3, "faux_positif": 3},
)
for nom, sous in exemples.items():
    print(nom, "->", len(sous), "exemple(s)")

## 3. Cartes Grad-CAM

In [ ]:
lignes = [(nom, r) for nom, sous in exemples.items() for _, r in sous.iterrows()]
fig, axes = plt.subplots(2, len(lignes), figsize=(2.6 * len(lignes), 5.4))
for k, (nom, r) in enumerate(lignes):
    img = ouvrir(r.image)
    x = t_eval(img)
    carte, proba = gradcam(modele, x, DEVICE)
    axes[0, k].imshow(img); axes[0, k].axis("off")
    axes[0, k].set_title(f"{HOPITAL_LABEL[r.center]}\n{nom}\np={proba:.2f}", fontsize=7)
    axes[1, k].imshow(img); axes[1, k].imshow(carte, cmap="jet", alpha=0.45); axes[1, k].axis("off")
plt.tight_layout()
plt.savefig("../outputs/figures/09_gradcam.png", dpi=130)
plt.show()

## 4. Sauvegarde

In [ ]:
metrics = {
    "variante": VARIANTE_RETENUE,
    "seuil_utilise": SEUIL,
    "exemples": [
        {"categorie": nom, "hopital": int(r.center), "image_id": int(r.image_id), "proba": float(r.proba)}
        for nom, r in lignes
    ],
}
print(save_metrics(metrics, "outputs/metrics/08_interpretabilite.json"))

## 5. Lecture

Voir `outputs/rapport/08_interpretabilite.md` : où se concentre l'activation sur chaque exemple (noyaux/amas cellulaires vs zones diffuses de fond ou de teinte), et ce que cela dit sur ce que le modèle a réellement appris.